# University of Engineering and Technology Peshawar, Nowshera Campus
## Lab 5: AEP Dataset - Normalization, One-Hot Encoding & Cyclic Features

**Course:** Machine Learning Lab  
**Student Name:** Sohaib ur Rehman Farooqi  
**Registration Number:** 22JZELE0458  
**Date:** June 24, 2026

---
**Objective:** Perform data normalization using MinMaxScaler, apply One-Hot Encoding on categorical features, and generate cyclic features for time-related columns.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import pickle
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from timeseires.utils import t_v_t_split as sp

## 2. Load Dataset

In [ ]:
df = pd.read_csv(r'C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\5_features_extracted.csv', 
                 index_col=['Datetime'], parse_dates=['Datetime'])
df.head()

## 3. Dataset Information

In [ ]:
df.info()

## 4. Train-Validation-Test Split

In [ ]:
train_set, validation_set, test_set = sp.t_v_t(df, 70, 20, 10)
print(train_set.shape)
print(validation_set.shape)
print(test_set.shape)

## 5. Normalization (MinMaxScaler)

In [ ]:
train_set_load = train_set['aep'].values.reshape(-1, 1)
validation_set_load = validation_set['aep'].values.reshape(-1, 1)
test_set_load = test_set['aep'].values.reshape(-1, 1)

scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(train_set_load)

scaled_train = scaler.transform(train_set_load)
scaled_val = scaler.transform(validation_set_load)
scaled_test = scaler.transform(test_set_load)

pickle.dump(scaler, open("AEPscaler.pkl", "wb"))
print(scaled_train.shape)

## 6. One-Hot Encoding

In [ ]:
train_setO = train_set.values
holiday = train_setO[:, 2:3]
weekend = train_setO[:, 3:4]

en_holiday = OneHotEncoder(handle_unknown='ignore')
en_weekend = OneHotEncoder(handle_unknown='ignore')

holidayt = en_holiday.fit_transform(holiday).toarray()
weekendt = en_weekend.fit_transform(weekend).toarray()

train_categorical = np.concatenate((holidayt, weekendt), axis=1)
print(train_categorical.shape)

## 7. Cyclic Features

In [ ]:
cyclic_train = train_set[['month','day_of_week','hour','winter','spring','summer','fall','year_day']].values

def cyclic_encode(data, period):
    return np.sin(2*np.pi*data/period), np.cos(2*np.pi*data/period)

sin_month, cos_month = cyclic_encode(cyclic_train[:,0:1], 12)
sin_week, cos_week = cyclic_encode(cyclic_train[:,1:2], 6)
sin_hour, cos_hour = cyclic_encode(cyclic_train[:,2:3], 24)
sin_winter, cos_winter = cyclic_encode(cyclic_train[:,3:4], 4)
sin_spring, cos_spring = cyclic_encode(cyclic_train[:,4:5], 4)
sin_summer, cos_summer = cyclic_encode(cyclic_train[:,5:6], 4)
sin_fall, cos_fall = cyclic_encode(cyclic_train[:,6:7], 4)
sin_year, cos_year = cyclic_encode(cyclic_train[:,7:8], 365)

train_cyclic = np.concatenate((sin_month, cos_month, sin_week, cos_week, sin_hour, cos_hour,
                               sin_winter, cos_winter, sin_spring, cos_spring, sin_summer, cos_summer,
                               sin_fall, cos_fall, sin_year, cos_year), axis=1)

## 8. Combine & Save Final Files

In [ ]:
train_final = np.concatenate((scaled_train, train_categorical, train_cyclic), axis=1)

columns = ['aep', 'Is_holiday1','Is_holiday2', 'Is_Weekend1','Is_Weekend2',
           'sin_month', 'cos_month', 'sin_week','cos_week', 'sin_hour', 'cos_hour',
           'sin_winter','cos_winter','sin_spring','cos_spring','sin_summer','cos_summer',
           'sin_fall','cos_fall','sin_year_day','cos_year_day']

train_df = pd.DataFrame(train_final, columns=columns)
train_df.to_csv('7_AEP_train.csv', index=False)
print("Train file saved successfully!")